## ***Initials***

In [1]:
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from collections import defaultdict
from geopy.distance import geodesic

## ***Help Functions***

In [2]:
def compare_sets(df1, df2):

    # Get common NACE codes between the two files
    common_nace = set(df1['NACE Code']).intersection(set(df2['NACE Code']))

    # Initialize stats
    any_common = 0
    full_match = 0
    overlap_counts = []

    for code in common_nace:
        row_a = df1[df1['NACE Code'] == code][['Top1', 'Top2', 'Top3']].values.flatten()
        row_b = df2[df2['NACE Code'] == code][['Top1', 'Top2', 'Top3']].values.flatten()

        set_a = {x for x in row_a if pd.notna(x)}
        set_b = {x for x in row_b if pd.notna(x)}

        intersection = set_a & set_b

        if intersection:
            any_common += 1
        if len(intersection) == 3:
            full_match += 1

        overlap_counts.append(len(intersection))

    # Final metrics
    total = len(common_nace)
    percent_any_common = any_common / total * 100
    percent_full_match = full_match / total * 100
    avg_overlap = sum(overlap_counts) / total

    print(f"Total common NACE codes: {total}")
    print(f"% with at least 1 common suggestion: {percent_any_common:.2f}%")
    print(f"% with all 3 suggestions matching: {percent_full_match:.2f}%")
    print(f"Average overlap count: {avg_overlap:.2f}")

## ***Load the Data***

In [4]:
# Load CSV files
df_supervised = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\1. Supervised Approach\\2. Approach 2 (won)\\Extracted CSV Files\\top10.csv")
df_unsupervised = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\2. Unsupervised Approach\\2. Approach 2 (won)\\Extracted CSV Files\\top10.csv")
df_semi_supervised = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\3. Semi-Supervised Approach\\1. Approach 1 (won)\\Extracted CSV Files\\top10.csv")

In [5]:
# Get union of all NACE codes across files
nace_codes = sorted(set(df_supervised['NACE Code']).union(df_unsupervised['NACE Code']).union(df_semi_supervised['NACE Code']))
print(f"length of NACE codes: {len(nace_codes)}")

length of NACE codes: 306


## ***Perform the Ensemble Method***

The weights per approach were estimated by their ***percentage of central suggestions***, since this is pre-decided as an important factor in general. Those were:

- Supervised Approach: 81.48%
- Unsupervised Approach: 93.79%
- Semi-Supervised Approach: 86.67%

In [6]:
centrality_scores = {
    'supervised': 0.8148,
    'unsupervised': 0.9379,
    'ssl': 0.8667
}


# # for the new clustering approach
# centrality_scores = {
#     'supervised': 0.8148,
#     'unsupervised': 0.9150,
#     'ssl': 0.8667
# }

In [7]:
total = sum(centrality_scores.values())
method_weights = {k: v / total for k, v in centrality_scores.items()}

In [8]:
method_weights

{'supervised': 0.3110636023516836,
 'unsupervised': 0.35805909750324505,
 'ssl': 0.33087730014507144}

In [13]:
position_weights = {
    f'Top{i}': 11 - i for i in range(1, 11)
}

In [14]:
position_weights

{'Top1': 10,
 'Top2': 9,
 'Top3': 8,
 'Top4': 7,
 'Top5': 6,
 'Top6': 5,
 'Top7': 4,
 'Top8': 3,
 'Top9': 2,
 'Top10': 1}

In [16]:
final_rows = []

for code in nace_codes:
    scores = defaultdict(float)

    for df, method_name in zip([df_supervised, df_unsupervised, df_semi_supervised], ['supervised', 'unsupervised', 'ssl']):
        row = df[df['NACE Code'] == code]
        if not row.empty:
            for pos in ['Top1', 'Top2', 'Top3', 'Top4', 'Top5', 'Top6', 'Top7', 'Top8', 'Top9', 'Top10']:
                value = row.iloc[0][pos]
                if pd.notna(value):
                    scores[value] += method_weights[method_name] * position_weights[pos]

    # Get top 3 communities by score
    top10 = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:10]
    top_names = [t[0] for t in top10]

    # Pad if fewer than 3
    while len(top_names) < 10:
        top_names.append(None)

    final_rows.append({
        'NACE Code': code,
        'Top1': top_names[0],
        'Top2': top_names[1],
        'Top3': top_names[2],
        'Top4': top_names[3],
        'Top5': top_names[4],
        'Top6': top_names[5],
        'Top7': top_names[6],
        'Top8': top_names[7],
        'Top9': top_names[8],
        'Top10': top_names[9],
    })
    
final_df = pd.DataFrame(final_rows)

In [17]:
print(final_df.shape)
final_df.head()

(306, 11)


,NACE Code,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,Top9,Top10
0,1.13,Metamorfosi,Kato Lehonia,Palaia,Epta Platania - Oksigono,Agioi Anargiroi,Agia Paraskeui,Agios Konstantinos,Agios Vasilios,Neapoli,Analipsi
1,1.30,Agia Paraskeui,Epta Platania - Oksigono,Neapoli,Agios Vasilios,Palaia,Nea Ionia,Agios Konstantinos,Asteria Agrias,Agia Kyriaki,Aivaliotika
2,1.49,Nees Pagases,Dimini,Melissatika,Agios Georgios,Neapoli,Epta Platania - Oksigono,Aivaliotika,Agios Vasilios,Alli Meria,Agia Paraskeui
3,1.61,Nees Pagases,Epta Platania - Oksigono,Dimitriada,Aivaliotika,Metamorfosi,Agia Paraskeui,Agios Georgios,Analipsi,Dimini,Agios Konstantinos
4,2.40,Metamorfosi,Analipsi,Asteria Agrias,Agios Konstantinos,Epta Platania - Oksigono,Agia Paraskeui,Agia Kyriaki,Palaia,Nees Pagases,Aivaliotika


## ***Save the File***

In [18]:
final_df.to_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\4. Ensembling of Approaches\\Extracted CSV Files\\top10_final.csv", index=False)

## ***Sanity Checks***

In [11]:
final_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\4. Ensembling of Approaches\\Extracted CSV Files\\top3_final.csv")

In [12]:
compare_sets(df_supervised, final_df)

Total common NACE codes: 306
% with at least 1 common suggestion: 98.04%
% with all 3 suggestions matching: 17.32%
Average overlap count: 1.97


In [13]:
compare_sets(df_unsupervised, final_df)

Total common NACE codes: 306
% with at least 1 common suggestion: 99.67%
% with all 3 suggestions matching: 19.93%
Average overlap count: 1.97


In [14]:
compare_sets(df_semi_supervised, final_df)

Total common NACE codes: 306
% with at least 1 common suggestion: 100.00%
% with all 3 suggestions matching: 16.01%
Average overlap count: 1.86
